What seperates an agent from a standard chatbot is its ability to take actions, percieve the output of those actions and react accordingly.

This is the method React agents use. Since most of the industry is coalesced around this method, from here on out we will just refer to React agents as Agents

NB: The actions that an agent can take are defined by the tools we provide it with.

Tools can allow our agent to access data, execute tasks, and even call other agents. This transofrms it from a passive language model to the coordinator of a much more capable system.

Let's define our tools, similar to how we defined our agent earlier on.

In [ ]:
from dotenv import load_dotenv # import load_dotenv

load_dotenv()

True

To define the tool we need to import 'tool' from langchain.tools.
Tools are defined like functions, but must be pre-empted by adding a decorator before the function.

This actually turns it from a function to a tool that the agent can use.

In [2]:
from langchain.tools import tool  #import tool from langchain.tools

@tool  #aforementioned tool decorator
def square_root(x: float) -> float:  #clearly define the tool
    """Calculate the square root of a number""" # clearly describe the tool to allow the agent to know the tools function
    return x ** 0.5

Alternatively, the tools name can be identified in the tool decorator

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

This can all however be overriden by writing both the tool name and description in the tool decorator

In [4]:
@tool("square_root", description='Calculate the square root of a number')
def tool1(x: float) -> float:
    return x ** 0.5

You could simply invoke the tool by passing the argument it expects to test.

This is how the agent would use it, just not hard coded

In [5]:
tool1.invoke({"x": 4879760})

2209.0178813219236

Using tools in agents.

When calling an agent, one needs to highlight the tools it has accessible to it, sometimes even including this in the system prompt to let it know what it has.

In [6]:
#import the necessary create_agent package
from langchain.agents import create_agent

#because we are using gemini, create first the model
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

#Create the agent
agent = create_agent(
    model=model,        # model created earlier
    tools =[tool1],    # a list of tools it has at its disposal
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
# We can use the human message package to have a question and answer system.
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467")

response=agent.invoke(
    {"messages":[question]}
)
print(response['messages'][-1].content)

The square root of 467 is 21.61.


Let's take a look under the hood to see what really went on.

In [10]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467', additional_kwargs={}, response_metadata={}, id='16addcc5-2063-4eee-a72c-6e461ded5634'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'square_root', 'arguments': '{"x": 467}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bd651-ed48-7fd0-83d9-eefd17148f56-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': '61082f5c-5491-43d9-bc90-4461ad2386fe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='21.61018278497431', name='square_root', id='31fd5898-52c7-4048-ba2d-06d3f126a5a9', tool_call_id='61082f5c-5491-43d9-bc90-4461ad2386fe'),
 AIMessage(content='The square root of 467 is 21.61.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'm

Notice that the ai message returned is blank (''). We could scroll and find the tool_calls field. Printing this out individually reveals the tool called, the arguments passed into it and the call id.

The response from the tool back to the agent is called the tool message (hitherto unseen) That just provides the answer from the tool and passes it back to the ai for a coherent response.

In [11]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': '61082f5c-5491-43d9-bc90-4461ad2386fe', 'type': 'tool_call'}]


Thsi is a rather trivial example as LLM's are able to compute mathematical operations such as this easily. Let's try to use a more realistic example: Web Scrapping (or simply searching the internet)

Let's create an agent with no tools and ask it a simple question.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

agent = create_agent(
    model=model, 
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [13]:

from langchain.messages import HumanMessage

question = HumanMessage(content="What is the population of Nairobi curently")

response=agent.invoke(
    {"messages":[question]}
)

In [14]:
print(response['messages'][-1].content)

As of **2023**, the estimated population of Nairobi is **around 4.9 million people**.

It's important to note that this is an **estimate**. Official census data is collected periodically, and the population of a rapidly growing city like Nairobi is constantly changing.

For the most precise, up-to-the-minute figures, you would typically refer to:

*   **The Kenya National Bureau of Statistics (KNBS):** They conduct the official census and publish demographic data.
*   **Reputable demographic research organizations:** These organizations often use sophisticated modeling to provide current estimates.


Notice we asked for current, up-to-date statistics, but recieved 'as of 2023'. This is due to the training data used that was up until 2023. 

This model therefore wont know anything after the end of that year out of the gate.

Let's then add a web search feature to our agent using Tavily.

In [15]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str,Any]:
    """Search the web for information"""
    return tavily_client.search(query)

web_search.invoke("What is the population of Nairobi curently")

{'query': 'What is the population of Nairobi curently',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://worldpopulationreview.com/cities/kenya/nairobi',
   'title': 'Nairobi Population 2026',
   'content': "Nairobi's 2026 population is now estimated at 6,002,290. In 1950, the population of Nairobi was 137,456. Nairobi has grown by 235,300 in the last year, which",
   'score': 0.9996947,
   'raw_content': None},
  {'url': 'https://www.macrotrends.net/global-metrics/cities/21711/nairobi/population',
   'title': 'Nairobi, Kenya Metro Area Population (1950-2026)',
   'content': 'The current metro area population of Nairobi in 2026 is 6,002,000, a 4.07% increase from 2025. · The metro area population of Nairobi in 2025 was 5,767,000, a',
   'score': 0.9996699,
   'raw_content': None},
  {'url': 'https://www.citieschange.org/cities/nairobi/',
   'title': 'Nairobi - CHANGE',
   'content': 'It is the most populous city in East Africa, with a current 

Notice that the response is a load of search results and articles from the web.

Let's add that tool to our agent so it has the most up-to-date information, and ask it the exact same question.

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

agent = create_agent(
    model=model,
    tools =[web_search]
)

question = HumanMessage(content= "What is the population of Nairobi currently?")

answer = agent.invoke(
    {"messages": [question]}
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [30]:
from pprint import pprint
pprint(answer['messages'])

[HumanMessage(content='What is the population of Nairobi currently?', additional_kwargs={}, response_metadata={}, id='a913b687-64a6-4872-8e8f-b2eb6932b01f'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "population of Nairobi currently"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bd669-433e-78b0-bf15-444f5393030d-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'population of Nairobi currently'}, 'id': '6365bf1f-103a-4aed-b7d6-cb69de8b5675', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 18, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"query": "population of Nairobi currently", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://worldpopulationreview.com/cities/kenya/nairobi"

These pprint statements seem however, a sub-optimal way to visualize how our agent runs.
That's where tracing on Langsmith comes in.

This is an optional recommendation that uses the langsmith API and the actual tracing can be seen on the langsmith website.